# Train Model 1 on your DRC data (local)

This notebook trains **Model 1**, the fast "keep this screen or not?" model, on the data your
DesktopResearchCollector (DRC) gathered on this PC. It runs the same steps as **Model 2**, the
learning loop in `adaptation/scheduled_job.py`, one at a time so you can see each step and its
result.

| Step | What happens | Who does the work |
|---|---|---|
| 1. Replay | Each DRC screenshot becomes one *capture* with what the live collector would have known at that moment: app, title, website, how long you'd been there, keyboard/mouse activity | `offline/drc_replay.py` |
| 2. Eyes | MobileCLIP turns each screenshot into 512 numbers that describe how the screen looks (frozen, never trained) | `features/mobileclip_encoder.py` |
| 3. Features | Each capture becomes one row of numbers, the model's inputs | `features/feature_assembly.py` |
| 4. Labels | "Did you come back to this within 48 h?" becomes the answer 1 or 0. **No hand labelling needed** | `adaptation/label_construction.py` |
| 5. Split | Older days train, the newest days test | `offline/training.py` |
| 6. Prototypes | Groups of important-looking screens (learned from training days only) | `adaptation/prototype_learning.py` |
| 7. Train | LightGBM learns inputs → answer | `models/lightgbm_model.py` |
| 8. Grade | Precision/recall, and storage saved compared with DRC keeping every screenshot | `eval/model_comparison.py` |
| 9. Hand labels | *(optional)* your own Y/N answers for a fair final grade | this notebook |
| 10. Save | Append-only model version, optionally activated for the live worker | `offline/training.py` |
| 11. Bundle | A copy for the Colab notebook, without images, titles or websites | `offline/training.py` |

**How to run:** in VS Code, open this file → *Select Kernel* → `python-worker/.venv`. Then *Run All*.
The first run spends about 5–10 minutes on step 2 (CPU); later runs reuse the cache and take
seconds.

**Safety:** `D:\DesktopResearchCollector` is only ever **read** (DRC can keep running). Everything this
notebook produces (embeddings, bundle, hand labels) goes to `WORK_DIR`, outside the repo, because
it is derived from your personal data.

> **Before committing this notebook:** *Clear All Outputs* first. After running, the outputs
> contain your screenshots, app names and window titles.

## 0. Settings
Change these if your paths differ. Nothing else in the notebook needs editing.

In [ ]:
import os
import sys
from pathlib import Path

DRC_FOLDER = Path(r"D:\DesktopResearchCollector")      # your DRC data (read-only)
TEST_FRACTION = 0.3        # newest 30% of labelled captures are the test set
PROTOTYPE_K = 8            # how many groups of "important-looking" screens to learn
THRESHOLD = 0.5            # Model 1 keeps a screen when its score >= THRESHOLD
SHOW_SCREENSHOTS = True    # show example screenshots per prototype (only on this PC)

SAVE_MODEL = True          # save the trained model as a new version (append-only)
DEPLOY_TO_LIVE_DB = False  # also activate it in python-worker/context.db if it beats what's live
REGISTER_EXISTING_VERSION = None  # e.g. a version trained in Colab and copied into models/versions/

EXPORT_BUNDLE = True       # write a privacy-reduced dataset for the Colab notebook
UPLOAD_BUNDLE = False      # copy that bundle to the Google Cloud bucket (needs gcloud login)
BUCKET = "desktop-research-data-collection"

LABEL_NOW = False          # set True to hand-label screenshots in step 9
N_TO_LABEL = 200


def _find_worker_dir() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base / "python-worker", base / "context-importance-system" / "python-worker"):
            if (candidate / "offline" / "training.py").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Open this notebook from inside the context-importance-system repo.")


WORKER_DIR = _find_worker_dir()
sys.path.insert(0, str(WORKER_DIR))

DRC_DB = DRC_FOLDER / "collector.db"
CHECKPOINT = Path(os.environ.get(
    "MOBILECLIP_S0_CHECKPOINT",
    WORKER_DIR.parents[1] / "Member1" / "checkpoints" / "mobileclip_s0.pt",
))
WORK_DIR = Path(os.environ.get("LOCALAPPDATA", Path.home())) / "context-importance" / "drc-training"
WORK_DIR.mkdir(parents=True, exist_ok=True)
LIVE_DB = WORKER_DIR / "context.db"
HAND_LABELS_CSV = WORK_DIR / "hand_labels.csv"

import torch  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("python-worker :", WORKER_DIR)
print("DRC database  :", DRC_DB, "(exists)" if DRC_DB.is_file() else "(MISSING)")
print("MobileCLIP    :", CHECKPOINT, "(exists)" if CHECKPOINT.is_file() else "(MISSING)")
print("work folder   :", WORK_DIR)
print("device        :", DEVICE)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from eval.model_comparison import AlwaysSaveScorer
from models.rule_based import RuleBasedImportanceScorer
from models.static_threshold import StaticThresholdScorer
from offline.drc_replay import replay_drc
from offline.training import (
    TEXT_FEATURE_KEYS, add_labels, build_dataset, embed_captures, evaluate_models, export_bundle,
    learn_prototypes, new_version_id, register_version, save_version, temporal_split,
    threshold_curve, train_model1, with_prototypes,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Replay the DRC data
Each DRC screenshot becomes one capture, described the way the live collector would have described
it at that moment. Captures are **dropped** (and counted below) for privacy and correctness:
- this project's exclusion list: password managers, login pages;
- the lock screen, which is time away, not a task;
- browser screenshots whose website is unknown. These fail closed, like the live collector.

In [ ]:
captures, replay_stats = replay_drc(DRC_DB)
print(replay_stats.summary())

caps = pd.DataFrame({"day": [c.ts.date() for c in captures], "app": [c.app for c in captures]})
display(caps.groupby("day").size().rename("captures per day").to_frame().T)
display(caps["app"].value_counts().head(10).rename("captures").to_frame())

## 2. The eyes: MobileCLIP embeddings
MobileCLIP is **not trained here**. Apple already trained it. It converts each screenshot into 512
numbers that describe how the screen looks (code editor? video? document?). Results are cached in
`WORK_DIR`, so this is slow only the first time.

In [ ]:
embeddings = embed_captures(captures, WORK_DIR / "drc_embeddings.npz", checkpoint=str(CHECKPOINT), device=DEVICE)
covered = sum(c.screenshot_id in embeddings for c in captures)
print(f"{covered} of {len(captures)} captures have an embedding")

## 3. Features: what Model 1 sees
Each capture becomes one row of numbers. This is the same code the live worker runs, replayed in
time order. Below is one example row; text fields are hidden because the model never uses them.
`rule_proba` is what today's hand-written rules would have scored it.

In [ ]:
dataset = build_dataset(captures, embeddings)
example = dataset.iloc[len(dataset) // 2]
display(pd.Series({k: v for k, v in example["features"].items() if k not in TEXT_FEATURE_KEYS}, name="value").to_frame())
print(f"rules would keep {(dataset['rule_decision'] == 'save').mean():.0%} of captures")

## 4. Labels: the answers, made automatically
For every capture: **did you come back to the same thing (same app + title + website) later, within
48 hours?** If you did, the label is 1 (important). If you didn't, it's 0. Captures from the last 48
hours stay unlabelled (*censored*), because their 48-hour window hasn't finished yet.

In [ ]:
dataset = add_labels(dataset)
labelled = dataset[dataset["label"].notna()]
print(f"labelled: {len(labelled)}   still waiting (last 48 h): {dataset['label'].isna().sum()}")
print(f"important (label 1): {int(labelled['label'].sum())} = {labelled['label'].mean():.0%}")
print(f"came back later: {int(labelled['revisited'].sum())},  came back in a later session: {int(labelled['continued'].sum())}")

by_app = labelled.groupby("app")["label"].agg(captures="size", important_share="mean")
display(by_app.sort_values("captures", ascending=False).head(12).style.format({"important_share": "{:.0%}"}))

## 5. Split by time: older days train, newest days test
Testing on the days that come *after* training mimics real use. The **embargo** drops training
captures whose 48-hour label window reaches into the test days. Otherwise their answers would leak
test-period behaviour into training.

In [ ]:
train, test = temporal_split(dataset, test_fraction=TEST_FRACTION)
for name, part in (("train", train), ("test", test)):
    print(f"{name:5}: {len(part):5} captures  {part['ts'].min():%d %b %H:%M} -> {part['ts'].max():%d %b %H:%M}"
          f"  important {part['label'].mean():.0%}")
assert train["label"].nunique() == 2, "training data needs both important and unimportant captures"

## 6. Prototypes: what important screens tend to look like
Model 2 groups the **training** captures that turned out important into `PROTOTYPE_K` clusters
(prototypes). Every capture then gets a score for how close it looks to the nearest prototype.
Model 1 can use that score, and this is how the eyes reach Model 1. Recomputing the score is
essential: at capture time there were no prototypes yet, so it was 0 everywhere.

In [ ]:
proto = learn_prototypes(train, k=PROTOTYPE_K)
prototypes = proto.centroids if proto is not None else None
if proto is None:
    print("Not enough important captures with embeddings to learn prototypes; visual similarity stays neutral.")
else:
    print(f"{proto.k} prototypes from {proto.n_samples} important training screenshots; sizes: {proto.counts}")

train_p, test_p = with_prototypes(train, prototypes), with_prototypes(test, prototypes)

In [ ]:
if SHOW_SCREENSHOTS and proto is not None:
    from PIL import Image

    important = train[(train["label"] == 1) & train["embedding"].notna()].reset_index(drop=True)
    sims = np.vstack(important["embedding"]) @ prototypes.T
    per_row = 4
    fig, axes = plt.subplots(proto.k, per_row, figsize=(per_row * 3, proto.k * 1.9))
    for k in range(proto.k):
        nearest = np.argsort(-sims[:, k])[:per_row]
        for j, idx in enumerate(nearest):
            ax = axes[k, j]
            with Image.open(important.loc[idx, "image_path"]) as im:
                ax.imshow(im.convert("RGB").resize((320, 180)))
            ax.set_axis_off()
            if j == 0:
                ax.set_title(f"prototype {k}: {important.loc[idx, 'app']}", fontsize=8, loc="left")
    plt.tight_layout()
    plt.show()

## 7. Train Model 1
LightGBM builds many small decision trees, each correcting the previous ones. Training takes
seconds. The chart shows which inputs Model 1 relied on most (*gain*).

In [ ]:
model, summary = train_model1(train_p)
print(f"trained on {summary['n_train']} captures, classes {summary['class_counts']}")

top = pd.Series(model.feature_importances("gain")).head(15)[::-1]
top.plot.barh(figsize=(7, 5), title="What Model 1 relies on (gain)")
plt.tight_layout()
plt.show()

## 8. Grade it on the test days
- **precision:** of the screens it kept, how many were really important (high = little wasted storage)
- **recall:** of the really important screens, how many it kept (high = little lost)
- **f1:** one number combining both
- **keep_rate:** share of screens kept
- **storage_reduction:** JPEG bytes saved compared with DRC, which keeps every screenshot

"Model 1 without eyes" is the same model trained with every visual feature blanked. The gap
between the two rows shows what MobileCLIP adds.

In [ ]:
def without_vision(df):
    out = df.copy()
    out["features"] = [{**f, **{k: 0 for k in f if k.startswith("vis_")}} for f in df["features"]]
    return out


model_no_eyes, _ = train_model1(without_vision(train_p))
models = {
    "Rules (what the live worker uses today)": RuleBasedImportanceScorer(),
    "Static: dwell >= 30 s": StaticThresholdScorer(),
    "Model 1 (LightGBM)": model,
    "Model 1 without eyes": model_no_eyes,
    "DRC: keep every screenshot": AlwaysSaveScorer(),
}
scored_test = {name: (without_vision(test_p) if name == "Model 1 without eyes" else test_p) for name in models}
results = pd.concat(
    [evaluate_models(scored_test[name], {name: m}, threshold=THRESHOLD) for name, m in models.items()],
    ignore_index=True,
)
display(results[["name", "precision", "recall", "f1", "keep_rate", "storage_reduction"]]
        .style.format({c: "{:.2f}" for c in ["precision", "recall", "f1", "keep_rate", "storage_reduction"]}))

### Choosing the threshold
A lower threshold keeps more screens (higher recall, less storage saved), and a higher threshold
keeps fewer. Use this chart to pick the trade-off you want, then set `THRESHOLD` in step 0.
*Caveat:* choosing it on the test days makes the test score slightly optimistic. With more data,
pick it on a separate validation period.

In [ ]:
curve = threshold_curve(test_p, model)
ax = curve.plot(x="threshold", y=["precision", "recall", "keep_rate", "storage_saved"], figsize=(7, 4), marker=".")
ax.axvline(THRESHOLD, color="grey", linestyle="--")
ax.set_ylim(0, 1.02)
ax.set_title("Model 1 on test days: trade-off by threshold")
plt.tight_layout()
plt.show()
display(curve.round(3).set_index("threshold").T)

## 9. (Optional) Your own answers: the fair final grade
Step 8 grades Model 1 against the automatic "came back within 48 h" labels, the same rule it was
trained on. To show it finds what matters to **you**, label a random sample of test screenshots
yourself:

1. Set `LABEL_NOW = True` in step 0 and run the next cell.
2. For each screenshot answer **y** (I'd want to find this again), **n** (no), **s** (skip) or
   **q** (quit, your progress is saved).
3. Answer **before** looking at what the model predicted.

Answers are saved to `hand_labels.csv` in `WORK_DIR`. They are only used for grading, never for
training.

In [ ]:
if LABEL_NOW:
    from IPython.display import clear_output
    from PIL import Image

    done = pd.read_csv(HAND_LABELS_CSV) if HAND_LABELS_CSV.exists() else pd.DataFrame(columns=["screenshot_id", "hand_label"])
    pool = test[~test["screenshot_id"].isin(done["screenshot_id"])]
    sample = pool.sample(n=min(max(N_TO_LABEL - len(done), 0), len(pool)), random_state=len(done))
    for i, row in enumerate(sample.itertuples(), start=1):
        clear_output(wait=True)
        with Image.open(row.image_path) as im:
            plt.figure(figsize=(12, 6.75))
            plt.imshow(im.convert("RGB"))
            plt.axis("off")
            plt.title(f"{row.app} | {row.title}"[:140], fontsize=9)
            plt.show()
        answer = input(f"[{len(done) + 1}/{N_TO_LABEL}] Would you want to find this again later? y/n/s/q: ").strip().lower()
        if answer == "q":
            break
        if answer in ("y", "n"):
            done.loc[len(done)] = [row.screenshot_id, 1 if answer == "y" else 0]
            done.to_csv(HAND_LABELS_CSV, index=False)
    clear_output()
    print(f"{len(done)} hand labels saved to {HAND_LABELS_CSV}")

In [ ]:
if HAND_LABELS_CSV.exists():
    hand = pd.read_csv(HAND_LABELS_CSV)
    graded = test_p.merge(hand, on="screenshot_id")
    if graded["hand_label"].nunique() == 2:
        scored_hand = {name: (without_vision(graded) if name == "Model 1 without eyes" else graded) for name in models}
        hand_results = pd.concat(
            [evaluate_models(scored_hand[n], {n: m}, threshold=THRESHOLD, label_column="hand_label") for n, m in models.items()],
            ignore_index=True,
        )
        agreement = (graded["hand_label"] == graded["label"].astype(int)).mean()
        print(f"{len(graded)} hand-labelled test screenshots; automatic labels agree with yours on {agreement:.0%}")
        display(hand_results[["name", "precision", "recall", "f1", "keep_rate"]].round(2))
    else:
        print("Hand labels need both 'y' and 'n' answers before they can grade a model.")
else:
    print("No hand labels yet (see the instructions above).")

## 10. Save the model
`SAVE_MODEL` stores Model 1 and its prototypes as a new version in `python-worker/models/versions/`.
Versions are never overwritten. `DEPLOY_TO_LIVE_DB` also records the version in the live worker's
`context.db` and activates it **only if it beats what the worker serves now**, which is the rule-based
baseline until a model is deployed. Restart the worker afterwards to load it.

In [ ]:
version_id = None
if SAVE_MODEL:
    version_id = new_version_id("lightgbm-drc")
    notes = {
        "source": str(DRC_DB), "replay": replay_stats.summary().splitlines(),
        "train": [str(train["ts"].min()), str(train["ts"].max()), len(train)],
        "test": [str(test["ts"].min()), str(test["ts"].max()), len(test)],
        "threshold": THRESHOLD, "results": results.drop(columns=["latency_mean_ms", "latency_p95_ms"]).to_dict("records"),
    }
    print("saved", save_version(model, proto, version_id, notes=notes))

if DEPLOY_TO_LIVE_DB:
    from models.lightgbm_model import ImportanceModel

    to_register = REGISTER_EXISTING_VERSION or version_id
    candidate = model if to_register == version_id else ImportanceModel.load(to_register)
    decision = register_version(LIVE_DB, candidate, test_p, to_register, threshold=THRESHOLD)
    print(("DEPLOYED: " if decision.deployed else "kept current model: ") + decision.reason)

## 11. Bundle for Colab
Writes the labelled dataset **without** screenshots, window titles, title words or websites (the
model never uses those). It keeps the numeric inputs, app name, label and the 512 numbers per
screenshot. `UPLOAD_BUNDLE` copies it to `gs://<BUCKET>/training-bundles/<participant>/`, where the
Colab notebook reads it.

In [ ]:
if EXPORT_BUNDLE:
    participant = json.loads((DRC_FOLDER / "collection_settings.json").read_text(encoding="utf-8"))["profile_id"]
    bundle = export_bundle(dataset, WORK_DIR / f"bundle-{participant}-{dataset['ts'].max():%Y%m%d}.parquet", participant=participant)
    print(f"bundle: {bundle} ({bundle.stat().st_size / 1e6:.1f} MB, {len(dataset)} rows)")
    if UPLOAD_BUNDLE:
        import subprocess

        target = f"gs://{BUCKET}/training-bundles/{participant}/{bundle.name}"
        subprocess.run(["gcloud", "storage", "cp", str(bundle), target], check=True, shell=(os.name == "nt"))
        print("uploaded to", target)

## What to do with the results
- **Model 1 beats "Rules" on f1** → the learning loop works on your data. Put the step 8 table in
  your report.
- **"without eyes" ≈ "Model 1"** → MobileCLIP isn't adding much yet. More data, or a different
  `PROTOTYPE_K`, may change that. Both outcomes are valid findings.
- **Low recall** → lower `THRESHOLD`. **Low storage reduction** → raise it.
- Re-run this notebook every few days as DRC collects more. Each run is a new, comparable version.